# Kalpa Health · Brief A · Part 1: Cleaning

Loads the merged legacy and Nova exports, patients and plan, applies the cleaning rules, and saves the clean tables that `Part 2: Analysis` reads.

**Run first.** Output: `h1_data.pkl`, `df_plan.pkl`, `df_patients.pkl` in `OUT_DIR`.

Cells marked `# ADDED` are new; everything else is your original code, in your original order.

In [ ]:
import pandas as pd
import numpy as np
import os

In [ ]:
DATA_DIR = r"/content/drive/MyDrive/Kalpa healthcare"

# ADDED: where the clean tables are saved for the analysis notebook
OUT_DIR = DATA_DIR


For Old_Legacy

In [ ]:
df_legacy = pd.read_csv(os.path.join(DATA_DIR, "old_legacy.csv"),low_memory=False)
print(df_legacy.head())

In [ ]:
#shape of data
df_legacy.shape

In [ ]:
#datatype of all columns
df_legacy.info()

In [ ]:
#convert dates
df_legacy['created_at']=pd.to_datetime(df_legacy['created_at'],format='%d/%m/%Y %H:%M',errors='coerce')
df_legacy['slot_at']=pd.to_datetime(df_legacy['slot_at'],format='%d/%m/%Y %H:%M',errors='coerce')

In [ ]:
# Convert numeric columns
df_legacy['test_price'] = pd.to_numeric(df_legacy['test_price'], errors='coerce')
df_legacy['home_fee'] = pd.to_numeric(df_legacy['home_fee'], errors='coerce')

In [ ]:
# Convert low-cardinality columns
categorical_columns = ['city', 'clinic_id', 'channel', 'source', 'test_code']
df_legacy[categorical_columns] = df_legacy[categorical_columns].astype('category')

In [ ]:
# Convert ID columns as strings
df_legacy['booking_id'] = df_legacy['booking_id'].astype('string')
df_legacy['patient_id'] = df_legacy['patient_id'].astype('string')
df_legacy['patient_name'] = df_legacy['patient_name'].astype('string')

In [ ]:
df_legacy.info()

In [ ]:
#missing values
df_legacy.isna().sum()

In [ ]:
#Remove NaN test code
df_legacy = df_legacy.dropna(subset=['test_price'])
df_legacy.isna().sum()

In [ ]:
# Duplicate rows
print("Duplicate rows:", df_legacy.duplicated().sum())

In [ ]:
#identifying duplicate rows
df_legacy.loc[df_legacy.duplicated()]

In [ ]:
#removing duplicate rows
df_legacy = df_legacy[df_legacy['booking_id']!='booking_id']
print("Duplicate rows:", df_legacy.duplicated().sum())

In [ ]:
#uniquely indentifying city
df_legacy['city'].unique()

In [ ]:
#Standardize city name
df_legacy["city"] = df_legacy["city"].replace("Bangalore", "Bengaluru")

In [ ]:
# Inspect one booking in the legacy system
booking_id = "BK-AHM-202501-000001"
df_legacy.loc[df_legacy["booking_id"] == booking_id]

In [ ]:
# Count rows and distinct tests per booking
legacy_booking_summary = (df_legacy.groupby("booking_id").agg(row_count=("booking_id", "size"),distinct_tests=("test_code", "nunique")))
legacy_booking_summary.head()

In [ ]:
#repeated test records
summary = df_legacy.groupby("booking_id").agg(row_count=("booking_id", "size"),distinct_tests=("test_code", "nunique"))
len(summary[summary["row_count"] > summary["distinct_tests"]])

In [ ]:
#Keep January to June 2025 and 2026
legacy_jan_jun = df_legacy[df_legacy["slot_at"].notna() & (df_legacy["slot_at"].dt.month <= 6) & (df_legacy["slot_at"].dt.year.isin([2025, 2026]))]

In [ ]:
#After cleaning
print("Legacy rows after basic cleaning:", len(legacy_jan_jun))
print("Date range:", legacy_jan_jun["slot_at"].min(), "to", legacy_jan_jun["slot_at"].max())
print(legacy_jan_jun["slot_at"].dt.year.value_counts().sort_index())

For Nova

In [ ]:
df_nova = pd.read_csv(os.path.join(DATA_DIR, "nova.csv"),low_memory=False)
print(df_nova.head())

In [ ]:
# Check the new system in the same way
df_nova.shape

In [ ]:
df_nova.info()

In [ ]:
#Convert date columns
#df_nova['created_at']

df_nova['created_at'] = pd.to_datetime(df_nova['created_at'].str[:16], format='%Y-%m-%d %H:%M', errors='coerce')
df_nova['slot_at']    = pd.to_datetime(df_nova['slot_at'].str[:16], format='%Y-%m-%d %H:%M', errors='coerce')

# Check: no date should fail to parse
print("created_at not parsed:", df_nova['created_at'].isna().sum())
print("slot_at not parsed:", df_nova['slot_at'].isna().sum())

In [ ]:
#Convert categorical columns
categorical_columns = ['city','clinic_id','channel','source','status_at_export','test_code']
df_nova[categorical_columns] = df_nova[categorical_columns].astype('category')

In [ ]:
#Convert ID columns to string
df_nova['booking_id'] = df_nova['booking_id'].astype('string')
df_nova['patient_id'] = df_nova['patient_id'].astype('string')
df_nova['rescheduled_from'] = df_nova['rescheduled_from'].astype('string')

In [ ]:
df_nova.info()

In [ ]:
#missing values
df_nova.isna().sum()

In [ ]:
# Duplicate rows
print("Duplicate rows:", df_nova.duplicated().sum())

In [ ]:
#uniquely indentifying city
df_nova['city'].unique()

In [ ]:
# Inspect one booking in the nova system
booking_id = "nova_che_0087556"
df_nova.loc[df_nova["booking_id"] == booking_id]

In [ ]:
# Count rows and distinct tests per booking
nova_booking_summary = (df_nova.groupby("booking_id").agg(row_count=("booking_id", "size"),distinct_tests=("test_code", "nunique")))
nova_booking_summary.head()

In [ ]:
#repeated test records
summary = df_nova.groupby("booking_id").agg(row_count=("booking_id", "size"),distinct_tests=("test_code", "nunique"))
len(summary[summary["row_count"] > summary["distinct_tests"]])

In [ ]:
is_rescheduled_original = df_nova["status_at_export"] == "rescheduled"

In [ ]:
# Inspect before removing
print("Rows on rescheduled originals:", is_rescheduled_original.sum())                        # expect 3,542
print("Rescheduled original bookings:", df_nova.loc[is_rescheduled_original, "booking_id"].nunique())  # expect 1,500

In [ ]:
df_nova = df_nova.loc[~is_rescheduled_original].copy()
print("Nova rows kept:", len(df_nova))

In [ ]:
nova_jan_jun = df_nova[df_nova["slot_at"].notna() & (df_nova["slot_at"].dt.month <= 6) & (df_nova["slot_at"].dt.year.isin([2025, 2026]))]

In [ ]:
#After cleaning
print("Nova rows after basic cleaning:", len(nova_jan_jun))
print("Date range:", nova_jan_jun["slot_at"].min(), "to", nova_jan_jun["slot_at"].max())
print(nova_jan_jun["slot_at"].dt.year.value_counts().sort_index())

Step 1: Combine Legacy and Nova

In [ ]:
legacy_jan_jun.head()

In [ ]:
nova_jan_jun.head()

In [ ]:
#select the columns needed to calculate test volume
legacy_h1 = legacy_jan_jun[["booking_id", "slot_at", "city", "channel", "patient_id", "test_code", "test_price"]]
nova_h1 = nova_jan_jun[["booking_id", "slot_at", "city", "channel", "patient_id", "test_code","test_price","rescheduled_from"]]

In [ ]:
print(len(legacy_h1))
print(len(nova_h1))

In [ ]:
# Legacy: every row is one test line, even if test_code is blank
legacy_h1["is_test"] = True
# Nova: a row is a test only if the booking's test list had a test (empty list -> blank test_code)
nova_h1["is_test"] = nova_h1["test_code"].notna()

In [ ]:
#add the same rescheduling column to Legacy
legacy_h1["rescheduled_from"] = pd.NA

In [ ]:
#Combine both systems into one dataset
h1_data = pd.concat([legacy_h1, nova_h1],ignore_index=True)
h1_data.head()

Step 2: Check the combined data



In [ ]:
# Check
print("Legacy tests:", legacy_h1["is_test"].sum())                                # expect 835,514
print("Nova tests:", nova_h1["is_test"].sum())                                    # expect 58,770
print("Nova empty bookings (0 tests):", (~nova_h1["is_test"]).sum())              # expect 20

In [ ]:
#check records by year
print("\nRecords by year:")
print(h1_data["slot_at"].dt.year.value_counts().sort_index())

In [ ]:
#create another column 'year'
h1_data['year']=h1_data['slot_at'].dt.strftime('%Y')
h1_data.head()

Step 3: Removing original bookings of rescheduled

In [ ]:
#count rows linked to an original booking
rescheduled_rows = nova_h1[nova_h1["rescheduled_from"].notna()]
print("Rescheduled rows:", len(rescheduled_rows))
rescheduled_rows.head()

In [ ]:
# Find original bookings referenced by Nova replacements
original_ids = rescheduled_rows["rescheduled_from"].unique()
print("Original booking IDs to check:", len(original_ids))

In [ ]:
#Remove those original bookings from Nova
h1_data = h1_data[~h1_data["booking_id"].isin(original_ids)]
h1_data.head()

### Channel labels (moved here from the H3 channel section)
Channel labels differ in case and spacing between the two systems, so they are standardised once here, not inside the analysis.

In [ ]:
h1_data['channel'] = (h1_data['channel'].str.strip().str.lower())

## Plan (cleaning only)

In [ ]:
df_plan = pd.read_csv(os.path.join(DATA_DIR, "plan_2026_H1.csv"),low_memory=False)
df_plan

In [ ]:
#shape of data
df_plan.shape

In [ ]:
#datatype of all columns
df_plan.info()

In [ ]:
#standadizing city and metric column
df_plan['city'] = df_plan['city'].str.strip()
df_plan['metric'] = df_plan['metric'].str.strip()

In [ ]:
#missing values
df_plan.isna().sum()

In [ ]:
#standardizing city names
df_plan['city']=df_plan['city'].replace('Bangalore','Bengaluru')

In [ ]:
df_plan

## Patients (cleaning only)

In [ ]:
df_patients = pd.read_csv(os.path.join(DATA_DIR, "patients.csv"),low_memory=False)
df_patients.head()

In [ ]:
df_patients.shape

In [ ]:
df_patients.info()

In [ ]:
# Convert date columns
df_patients['dob'] = pd.to_datetime(df_patients['dob'], errors='coerce')

df_patients['signup_date'] = pd.to_datetime(df_patients['signup_date'], errors='coerce')

In [ ]:
#Convert into strings
df_patients['patient_id'] = df_patients['patient_id'].astype('string')
df_patients['patient_name'] = df_patients['patient_name'].astype('string')
df_patients['phone'] = df_patients['phone'].astype('string')
df_patients['city'] = df_patients['city'].astype('string')

In [ ]:
df_patients.info()

In [ ]:
#missing_values
df_patients.isna().sum()

In [ ]:
df_patients['city'].unique()

In [ ]:
df_patients[['dob', 'signup_date']].head(10)

## Checks and hand-off

In [ ]:
# ADDED: checks on the clean combined table
print("h1_data rows:", len(h1_data))
print("Tests per year:")
print(h1_data.groupby("year")["is_test"].sum())      # expect 436,302 (2025) and 457,982 (2026)
print("Cities:", sorted(h1_data["city"].dropna().unique()))
print("Channels:", sorted(h1_data["channel"].dropna().unique()))
print("Duplicate patient_id in df_patients:", df_patients["patient_id"].duplicated().sum())


In [ ]:
# ADDED: save the clean tables for the analysis notebook
h1_data.to_pickle(os.path.join(OUT_DIR, "h1_data.pkl"))
df_plan.to_pickle(os.path.join(OUT_DIR, "df_plan.pkl"))
df_patients.to_pickle(os.path.join(OUT_DIR, "df_patients.pkl"))
print("Saved h1_data, df_plan, df_patients to", OUT_DIR)
